# Bronze — Typed Ingestion
Reads the 13 CaboodleProviderRaw CSVs with explicit schemas into `bronze_*` Delta tables.

In [ ]:
# ============================================================
# NOTEBOOK 1: BRONZE  —  CaboodleProviderRaw CSVs -> typed Delta
# ============================================================
from pyspark.sql import functions as F, types as T

RAW = "Files/raw/caboodle_provider"     # where the OneLake workflow landed the CSVs
BRONZE_PREFIX = "bronze_"

# ---- explicit schemas (codes = string, money = decimal, keys = int) ----
S = T.StructType
f = T.StructField
INT, STR, DBL, BOOL, DT = T.IntegerType(), T.StringType(), T.DoubleType(), T.BooleanType(), T.DateType()
MONEY = T.DecimalType(12, 2)

schemas = {
  "DimProvider": S([f("ProviderKey",INT),f("ProviderDurableKey",INT),f("NPI",STR),f("ProviderName",STR),
      f("FirstName",STR),f("LastName",STR),f("Credentials",STR),f("Gender",STR),f("ProviderType",STR),
      f("IsProviderActive",BOOL),f("PrimarySpecialty",STR),f("ProviderStatus",STR),f("HireDate",DT),
      f("TerminationDate",DT),f("_IsCurrent",BOOL),f("EffectiveDate",DT),f("ExpirationDate",DT)]),
  "DimProviderSpecialty": S([f("ProviderSpecialtyKey",INT),f("ProviderKey",INT),f("ProviderDurableKey",INT),
      f("SpecialtyName",STR),f("TaxonomyCode",STR),f("IsPrimary",BOOL),f("_IsCurrent",BOOL),
      f("EffectiveDate",DT),f("ExpirationDate",DT)]),
  "DimProviderCredential": S([f("ProviderCredentialKey",INT),f("ProviderKey",INT),f("ProviderDurableKey",INT),
      f("LicenseNumber",STR),f("LicenseState",STR),f("DEANumber",STR),f("BoardCertification",STR),
      f("LicenseExpirationDate",DT),f("DEAExpirationDate",DT),f("BoardCertExpirationDate",DT),
      f("_IsCurrent",BOOL),f("EffectiveDate",DT),f("ExpirationDate",DT)]),
  "ProviderDepartmentBridge": S([f("ProviderDepartmentKey",INT),f("ProviderKey",INT),f("DepartmentKey",INT),
      f("IsPrimaryDepartment",BOOL),f("_IsCurrent",BOOL),f("EffectiveDate",DT),f("ExpirationDate",DT)]),
  "DimDepartment": S([f("DepartmentKey",INT),f("DepartmentDurableKey",INT),f("DepartmentID",STR),
      f("DepartmentName",STR),f("ServiceLine",STR),f("DepartmentSpecialty",STR),f("_IsCurrent",BOOL),
      f("EffectiveDate",DT),f("ExpirationDate",DT)]),
  "DimFacility": S([f("FacilityKey",INT),f("FacilityDurableKey",INT),f("FacilityID",STR),f("FacilityName",STR),
      f("AddressLine1",STR),f("City",STR),f("StateAbbr",STR),f("ZIP",STR),f("Region",STR),
      f("_IsCurrent",BOOL),f("EffectiveDate",DT),f("ExpirationDate",DT)]),
  "DimPayer": S([f("PayerKey",INT),f("PayerDurableKey",INT),f("PayerID",STR),f("PayerName",STR),
      f("PlanType",STR),f("LineOfBusiness",STR),f("_IsCurrent",BOOL),f("EffectiveDate",DT),f("ExpirationDate",DT)]),
  "DimDiagnosis": S([f("DiagnosisKey",INT),f("DiagnosisDurableKey",INT),f("ICD10Code",STR),f("DiagnosisName",STR),
      f("DiagnosisCategory",STR),f("_IsCurrent",BOOL),f("EffectiveDate",DT),f("ExpirationDate",DT)]),
  "DimProcedure": S([f("ProcedureKey",INT),f("ProcedureDurableKey",INT),f("ProcedureCode",STR),f("CodeType",STR),
      f("ProcedureDescription",STR),f("_IsCurrent",BOOL),f("EffectiveDate",DT),f("ExpirationDate",DT)]),
  "DimPatient": S([f("PatientKey",INT),f("PatientDurableKey",INT),f("MRN",STR),f("FirstName",STR),f("LastName",STR),
      f("PatientName",STR),f("DateOfBirth",DT),f("Gender",STR),f("Race",STR),f("Ethnicity",STR),f("ZIP",STR),
      f("PCPProviderKey",INT),f("_IsCurrent",BOOL),f("EffectiveDate",DT),f("ExpirationDate",DT)]),
  "FactEncounter": S([f("EncounterKey",INT),f("PatientKey",INT),f("AttendingProviderKey",INT),
      f("ReferringProviderKey",INT),f("DepartmentKey",INT),f("LocationKey",INT),f("DiagnosisKey",INT),
      f("EncounterDate",DT),f("EncounterType",STR),f("LengthOfStay",INT)]),
  "FactClaim": S([f("ClaimKey",INT),f("PatientKey",INT),f("BillingProviderKey",INT),f("RenderingProviderKey",INT),
      f("PayerKey",INT),f("ProcedureKey",INT),f("DiagnosisKey",INT),f("ServiceDate",DT),
      f("BilledAmount",MONEY),f("AllowedAmount",MONEY),f("PaidAmount",MONEY),f("ClaimStatus",STR)]),
  "FactRiskScore": S([f("RiskScoreKey",INT),f("PatientKey",INT),f("ProviderKey",INT),f("RiskModel",STR),
      f("RiskScore",DBL),f("ScoreDate",DT)]),
}

# Spark's CSV reader only accepts true/false for BooleanType; the CaboodleProviderRaw
# CSVs write flags as 1/0 (Y/N, t/f), which parse to NULL. Read boolean columns as
# strings and convert them robustly so _IsCurrent et al. survive the load.
def _truthy(colname):
    c = F.upper(F.trim(F.col(colname).cast("string")))
    return c.isin("TRUE", "T", "1", "Y", "YES").cast("boolean")

for name, schema in schemas.items():
    bool_cols = [fld.name for fld in schema.fields if isinstance(fld.dataType, T.BooleanType)]
    read_schema = T.StructType([
        T.StructField(fld.name,
                      T.StringType() if isinstance(fld.dataType, T.BooleanType) else fld.dataType,
                      fld.nullable)
        for fld in schema.fields
    ])
    df = (spark.read.option("header", True).option("multiLine", True).option("escape", '"')
              .schema(read_schema).csv(f"{RAW}/{name}.csv"))
    for bc in bool_cols:
        df = df.withColumn(bc, _truthy(bc))
    df = (df.withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.lit(f"{name}.csv")))

    # Guard rail: fail LOUDLY if a boolean flag parsed to all-NULL (the silent
    # bug that emptied every silver table). Better to stop here than poison silver.
    df = df.cache()
    n = df.count()
    assert n > 0, f"{name}.csv loaded 0 rows — check RAW path '{RAW}' and file name."
    for bc in bool_cols:
        non_null = df.where(F.col(bc).isNotNull()).limit(1).count()
        assert non_null > 0, (
            f"{name}.{bc} parsed to all-NULL — unrecognized boolean token in the CSV. "
            f"Extend _truthy() with the raw value before re-running."
        )

    tbl = f"{BRONZE_PREFIX}{name.lower()}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(tbl)
    print(f"{tbl:35s} {n:>8,} rows")
    df.unpersist()
print("BRONZE complete.")